<a href="https://colab.research.google.com/github/Saransh-Basu-01/50-Ml-Projects-to-understand-LLMs/blob/main/Project_36_Characterstics_of_Attention_Adjustment_Magnitudes.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.gridspec import GridSpec

from datasets import load_dataset

import torch
from transformers import AutoModelForCausalLM, GPT2Tokenizer
import torch.nn.functional as F

# Part 1: Model, tokens, projection vectors

## Task_1:
- Import GPT-2-large and its tokenizer. Define a variable for the
number of transformer layers.
Create and implant hook functions into all attn.c_proj layers
that detaches the output of the attention block and stores it in a
dictionary with keys indicating the layer from which the vectors were
taken.
The online code file shows how to import the test set from
the “Rotten Tomatoes” dataset via the Hugging Face function
load_dataset. Explore the dataset by examining the number of
samples it contains and by printing out an example text sample.

In [2]:
# load GPT2 model and tokenizer
model = AutoModelForCausalLM.from_pretrained('gpt2-large')
tokenizer = GPT2Tokenizer.from_pretrained('gpt2')

model.eval()


config.json:   0%|          | 0.00/666 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 3.25GB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/436 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/1.04M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

GPT2LMHeadModel(
  (transformer): GPT2Model(
    (wte): Embedding(50257, 1280)
    (wpe): Embedding(1024, 1280)
    (drop): Dropout(p=0.1, inplace=False)
    (h): ModuleList(
      (0-35): 36 x GPT2Block(
        (ln_1): LayerNorm((1280,), eps=1e-05, elementwise_affine=True, bias=True)
        (attn): GPT2Attention(
          (c_attn): Conv1D(nf=3840, nx=1280)
          (c_proj): Conv1D(nf=1280, nx=1280)
          (attn_dropout): Dropout(p=0.1, inplace=False)
          (resid_dropout): Dropout(p=0.1, inplace=False)
        )
        (ln_2): LayerNorm((1280,), eps=1e-05, elementwise_affine=True, bias=True)
        (mlp): GPT2MLP(
          (c_fc): Conv1D(nf=5120, nx=1280)
          (c_proj): Conv1D(nf=1280, nx=5120)
          (act): NewGELUActivation()
          (dropout): Dropout(p=0.1, inplace=False)
        )
      )
    )
    (ln_f): LayerNorm((1280,), eps=1e-05, elementwise_affine=True, bias=True)
  )
  (lm_head): Linear(in_features=1280, out_features=50257, bias=False)
)

In [3]:
n_layers = model.config.n_layer

In [4]:
n_layers

36

In [7]:
# hook the attention vectors
activations = {}

def implant_hook(layer_number):
  def hook(module,input,output):
    activations[f'attn_L{layer_number}'] = output.detach()
  return hook

In [8]:
# implant the hooks
handles = []
for i in range(n_layers):
  h = model.transformer.h[i].attn.c_proj.register_forward_hook(implant_hook(i))
  handles.append(h)

In [9]:
# load a dataset
dataset = load_dataset('cornell-movie-review-data/rotten_tomatoes', split='test')
dataset

README.md:   0%|          | 0.00/7.46k [00:00<?, ?B/s]

train.parquet: reconstructing file:   0%|          |  0.00B /  699kB            

train.parquet: downloading bytes:           |  0.00B            

validation.parquet: reconstructing file:   0%|          |  0.00B / 90.0kB            

validation.parquet: downloading bytes:           |  0.00B            

test.parquet: reconstructing file:   0%|          |  0.00B / 92.2kB            

test.parquet: downloading bytes:           |  0.00B            

Generating train split:   0%|          | 0/8530 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/1066 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/1066 [00:00<?, ? examples/s]

Dataset({
    features: ['text', 'label'],
    num_rows: 1066
})

In [10]:
dataset[30]

{'text': "what's so striking about jolie's performance is that she never lets her character become a caricature -- not even with that radioactive hair .",
 'label': 1}

In [15]:
dataset[857]

{'text': "this is sandler running on empty , repeating what he's already done way too often .",
 'label': 0}

## Task_2:
- Create a batch of 64 text sequences. The movie reviews have varying
lengths, which means you’ll need to use padding to create a batch.
Remember that you need to define a pad token in the GPT-2
tokenizer object. Also remember that the shorter token sequences
are padded to the length of the longest token sequence in the batch.
Because different sequences have different lengths, you will need a
Boolean mask of size batch × tokens that contains True for valid
tokens, and False for padding tokens. This is different from the
attention_mask variable produced by the tokenizer, and will be used
later in the project. Visualize the Boolean mask using plt.pcolor
as in Figure 6.30A, and a histogram of sequence lengths as in Figure
6.30B